In [2]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

connection_string = (
    "mssql+pyodbc://@localhost/mlb?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(connection_string)

In [5]:
query = """
select * from mlb.dbo.fact_hitter_pitcher_matchup_model_features
"""

df = pd.read_sql(query, engine)

print(df.shape)
df.head()

(24609, 448)


,gamePk,game_date,season,hitter_id,hitter_name,hitter_position,hitter_team_id,hitter_team_name,pitcher_id,pitcher_name,...,pitcher_avg_chase_rate_last_10,pitcher_avg_zone_rate_last_10,pitcher_avg_velocity_last_10,pitcher_avg_spin_rate_last_10,pitcher_avg_sl_whiff_rate_last_10,pitcher_avg_ff_whiff_rate_last_10,pitcher_prev_whiff_rate,pitcher_prev_csw_rate,pitcher_prev_chase_rate,pitcher_strikeOuts
0,778032,2025-05-06,2025,672695,Geraldo Perdomo,SS,109,Arizona Diamondbacks,656849,David Peterson,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
1,777013,2025-07-25,2025,666181,Will Benson,LF,113,Cincinnati Reds,641793,Zack Littell,...,0.282231,0.534879,87.684055,1827.276940,0.078336,0.122787,0.112360,0.235955,0.306122,2
2,776712,2025-08-16,2025,672580,Maikel Garcia,3B,118,Kansas City Royals,680732,Sean Burke,...,0.320924,0.493231,88.397858,2492.036148,0.124928,0.119645,0.102273,0.306818,0.324324,3
3,776152,2025-09-27,2025,669369,Bryce Johnson,CF,135,San Diego Padres,694851,Andrew Hoffmann,...,0.263626,0.448639,90.804374,1635.216126,0.230159,0.117464,0.090909,0.181818,0.181818,0
4,777731,2025-05-28,2025,671289,Tyler Freeman,RF,115,Colorado Rockies,571510,Matthew Boyd,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8


In [7]:
# How many strikeouts the pitcher gets in THIS game
print(df.columns.tolist())

df_model = df.copy()

['gamePk', 'game_date', 'season', 'hitter_id', 'hitter_name', 'hitter_position', 'hitter_team_id', 'hitter_team_name', 'pitcher_id', 'pitcher_name', 'pitcher_team_id', 'pitcher_team_name', 'pitcher_throws', 'hitter_stand', 'hitter_strikeOuts', 'pitches_seen_vs_pitcher', 'swings_vs_pitcher', 'whiffs_vs_pitcher', 'called_strikes_vs_pitcher', 'matchup_whiff_rate', 'matchup_called_strike_rate', 'matchup_csw_rate', 'hitter_days_since_last_game', 'hitter_avg_k_last_3', 'hitter_avg_pa_last_3', 'hitter_avg_ab_last_3', 'hitter_avg_hits_last_3', 'hitter_avg_hr_last_3', 'hitter_avg_bb_last_3', 'hitter_avg_pitches_last_3', 'hitter_avg_tb_last_3', 'hitter_avg_rbi_last_3', 'hitter_avg_lob_last_3', 'hitter_avg_obp_last_3', 'hitter_avg_slg_last_3', 'hitter_avg_ops_last_3', 'hitter_avg_babip_last_3', 'hitter_avg_batting_avg_last_3', 'hitter_avg_hbp_last_3', 'hitter_avg_sf_last_3', 'hitter_avg_sbunts_last_3', 'hitter_avg_stolen_bases_last_3', 'hitter_avg_caught_stealing_last_3', 'hitter_avg_k_rate_last_

In [16]:
target = "pitcher_strikeOuts"

drop_cols = [
    "gamePk",
    "game_date",
    "hitter_name",
    "pitcher_name",
    "hitter_position",
    "hitter_team_name",
    "pitcher_team_name"
]

X = df.drop(columns=drop_cols + [target])
y = df[target]

print(x.shape)
print(y.shape)
print(X.dtypes[X.dtypes == "object"])

(24609, 443)
(24609,)
Series([], dtype: object)


In [17]:
# Handle categorical columns
# pitcher_thros (L/R) & hitter_stand (L/R)
# XGBoost cannot handle strings
# This will make pitcher_throws_R = 1 -> right handed
# pitcher_thros_R = 0 -> left handed

X = pd.get_dummies(X, columns=["pitcher_throws", "hitter_stand"], drop_first=True)

In [18]:
# clean nulls and fill it with 0
X = X.fillna(0)

In [19]:
# important - train 80% and test 20% (unseen data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size = 0.2,
    random_state = 42
)

print(X_train.shape, X_test.shape)

(19687, 440) (4922, 440)


In [ ]:
# Train XGBoost model - make sure there is no string in any of the columns
model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [22]:
# Predicit 
y_pred = model.predict(X_test)

print(y_pred[:10])

[6.5369325 3.991674  6.3926363 5.0490212 8.604009  4.33734   2.823751
 3.0149028 2.7390902 4.043201 ]


In [ ]:
# Metrics - measure accuracy 
# MAE - average error  0.8 = VERY GOOD, 1.2 = good, 1.5+ needs improvement
# RMSE - penalizes big misses, should be slightly higher than MAE

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 1.2418639659881592
RMSE: 1.5924689436299162


In [30]:
# Results table - 1 row = hitter vs pitcher

results = df.loc[X_test.index].copy()

results["predicted_strikeouts"] = y_pred

results[[
    "pitcher_name",
    "hitter_name",
    "pitcher_team_name",
    "pitcher_strikeOuts",
    "predicted_strikeouts",
]].head(20)

,pitcher_name,hitter_name,pitcher_team_name,pitcher_strikeOuts,predicted_strikeouts
3733,Kyle Bradish,Jarren Duran,Baltimore Orioles,10,6.536932
11502,Colin Rea,Ben Williamson,Chicago Cubs,2,3.991674
6423,Jesús Luzardo,Pedro Pagés,Philadelphia Phillies,6,6.392636
15983,Cade Povich,Ryan O'Hearn,Baltimore Orioles,3,5.049021
23613,Dylan Cease,Will Smith,Toronto Blue Jays,9,8.604009
21992,Taijuan Walker,Casey Schmitt,Philadelphia Phillies,3,4.337340
5843,Aaron Nola,CJ Abrams,Philadelphia Phillies,4,2.823751
2168,Antonio Senzatela,Luis García Jr.,Colorado Rockies,0,3.014903
12110,Ranger Suarez,Xander Bogaerts,Boston Red Sox,2,2.739090
1103,Andrew Abbott,Alec Burleson,Cincinnati Reds,3,4.043201


In [36]:
# Aggregate predictions - multiple hitters per pitch

pitcher_preds = results.groupby(
    ["gamePk", "pitcher_name","pitcher_team_name"]
).agg(
    actual_K=("pitcher_strikeOuts", "first"),
    predicted_K=("predicted_strikeouts", "mean")
).reset_index()

pitcher_preds.head(20)

,gamePk,pitcher_name,pitcher_team_name,actual_K,predicted_K
0,776136,Brandon Pfaadt,Arizona Diamondbacks,5,4.223785
1,776137,Antonio Senzatela,Colorado Rockies,0,1.561338
2,776137,Juan Mejia,Colorado Rockies,1,1.363863
3,776137,Logan Webb,San Francisco Giants,8,7.546691
4,776138,Cole Ragans,Kansas City Royals,8,7.286646
5,776140,Rico Garcia,Baltimore Orioles,2,0.743285
6,776141,Simeon Woods Richardson,Minnesota Twins,9,5.464324
7,776143,Brad Lord,Washington Nationals,4,4.894092
8,776143,Shane Smith,Chicago White Sox,8,6.596539
9,776144,Aaron Ashby,Milwaukee Brewers,2,1.716247


In [37]:
# add betting decision 
pitcher_preds["line"] = 5.5  # example (later from sportsbook)

pitcher_preds["edge"] = pitcher_preds["predicted_K"] - pitcher_preds["line"]

pitcher_preds["bet"] = pitcher_preds["edge"].apply(
    lambda x: "OVER" if x > 0 else "UNDER"
)

pitcher_preds.head(20)

,gamePk,pitcher_name,pitcher_team_name,actual_K,predicted_K,line,edge,bet
0,776136,Brandon Pfaadt,Arizona Diamondbacks,5,4.223785,5.5,-1.276215,UNDER
1,776137,Antonio Senzatela,Colorado Rockies,0,1.561338,5.5,-3.938662,UNDER
2,776137,Juan Mejia,Colorado Rockies,1,1.363863,5.5,-4.136137,UNDER
3,776137,Logan Webb,San Francisco Giants,8,7.546691,5.5,2.046691,OVER
4,776138,Cole Ragans,Kansas City Royals,8,7.286646,5.5,1.786646,OVER
5,776140,Rico Garcia,Baltimore Orioles,2,0.743285,5.5,-4.756715,UNDER
6,776141,Simeon Woods Richardson,Minnesota Twins,9,5.464324,5.5,-0.035676,UNDER
7,776143,Brad Lord,Washington Nationals,4,4.894092,5.5,-0.605908,UNDER
8,776143,Shane Smith,Chicago White Sox,8,6.596539,5.5,1.096539,OVER
9,776144,Aaron Ashby,Milwaukee Brewers,2,1.716247,5.5,-3.783753,UNDER


In [38]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

,feature,importance
331,pitcher_gamesStarted,0.217931
374,pitcher_avg_k_last_10,0.034023
8,whiffs_vs_pitcher,0.019425
356,pitcher_avg_k_last_5,0.017574
5,hitter_strikeOuts,0.017028
425,pitcher_avg_whiff_rate_last_10,0.014985
12,matchup_csw_rate,0.010233
415,pitcher_avg_whiff_rate_last_5,0.008949
10,matchup_whiff_rate,0.008941
389,pitcher_prev_bf,0.008517


In [39]:
feature_importance.head(20)

,feature,importance
331,pitcher_gamesStarted,0.217931
374,pitcher_avg_k_last_10,0.034023
8,whiffs_vs_pitcher,0.019425
356,pitcher_avg_k_last_5,0.017574
5,hitter_strikeOuts,0.017028
425,pitcher_avg_whiff_rate_last_10,0.014985
12,matchup_csw_rate,0.010233
415,pitcher_avg_whiff_rate_last_5,0.008949
10,matchup_whiff_rate,0.008941
389,pitcher_prev_bf,0.008517
